# LangChain ToolRuntime: Stream Writer, Execution Info & Server Info


## Topics
1. **Stream writer** — emit progress updates from a running tool.
2. **Execution info** — inspect thread ID, run ID, and node attempt.
3. **Server info** — inspect LangGraph Server assistant, graph, and authenticated-user metadata when available.

> Requirements: set `OPENAI_API_KEY`. `execution_info` requires `langgraph>=1.1.5` (or the documented compatible Deep Agents version). `server_info` is only populated when running on LangGraph Server; it is normally `None` in local development.

## 1. Stream writer — send live progress updates

### Concept
`runtime.stream_writer` lets a tool emit custom updates while it is running. These updates are separate from the tool's final return value.

### Code

In [ ]:
%pip install -U langchain langchain-openai "langgraph>=1.1.5"

In [ ]:

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_openai import ChatOpenAI


In [ ]:
@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Get weather for a city and stream progress updates."""

    writer = runtime.stream_writer

    writer(f"Looking up data for city: {city}")

    # Demo-only stand-in for a slow external API call.
    # Replace this with your actual weather API request.
    weather = f"It's always sunny in {city}!"

    writer(f"Acquired data for city: {city}")

    return weather

In [ ]:
stream_agent = create_agent(
    model=ChatOpenAI(model="gpt-5.5"),
    tools=[get_weather],
    system_prompt=(
        "You are a weather assistant. Use get_weather when asked about "
        "the weather. Report the tool result accurately."
    ),
)

### Run and observe streamed events

`stream_mode="custom"` exposes updates emitted by `runtime.stream_writer`. The agent is invoked inside the LangGraph execution context, which is required for the writer.

In [ ]:
for chunk in stream_agent.stream(
    {"messages": [{"role": "user", "content": "What's the weather in Pune?"}]},
    stream_mode="custom",
):
    print("CUSTOM UPDATE:", chunk)

## 2. Execution info — inspect the current run

### Concept
`runtime.execution_info` provides execution metadata such as:
- `thread_id` — conversation/thread identifier, when available.
- `run_id` — current run identifier.
- `node_attempt` — current node attempt/retry information.

This example prints the metadata from inside the tool. Values can depend on the execution context.

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def log_execution_context(runtime: ToolRuntime) -> str:
    """Log execution identity and attempt information."""

    info = runtime.execution_info

    print(f"Thread: {info.thread_id}")
    print(f"Run: {info.run_id}")
    print(f"Attempt: {info.node_attempt}")

    return "Execution information logged."

In [ ]:
execution_agent = create_agent(
    model=ChatOpenAI(model="gpt-5.5"),
    tools=[log_execution_context],
    system_prompt=(
        "When asked to inspect execution information, call "
        "log_execution_context and report that it completed."
    ),
)

In [ ]:
execution_result = execution_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Log your execution information."}
        ]
    },
    config={
        "configurable": {
            "thread_id": "execution-info-demo"
        }
    },
)

print(execution_result["messages"][-1].content)

## 3. Server info — inspect LangGraph Server metadata

### Concept
`runtime.server_info` can provide server-side metadata:
- `assistant_id`
- `graph_id`
- authenticated `user` information, when available

**Important:** `server_info` is `None` in local development/testing. Therefore, this local notebook demonstrates the safe `None` check; it cannot produce real assistant/graph/user server metadata unless deployed and invoked on LangGraph Server.

In [ ]:
@tool
def get_assistant_scoped_data(runtime: ToolRuntime) -> str:
    """Inspect assistant and authenticated-user metadata when on LangGraph Server."""

    server = runtime.server_info

    if server is None:
        return (
            "Server information is unavailable. "
            "This tool is not running on LangGraph Server."
        )

    print(f"Assistant: {server.assistant_id}")
    print(f"Graph: {server.graph_id}")

    if server.user is not None:
        print(f"User: {server.user.identity}")

    return "Server information inspected."

In [ ]:
server_info_agent = create_agent(
    model=ChatOpenAI(model="gpt-5.5"),
    tools=[get_assistant_scoped_data],
    system_prompt=(
        "When asked about server metadata, call "
        "get_assistant_scoped_data. Do not invent server information."
    ),
)

In [ ]:
server_result = server_info_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Inspect the server information available to you."}
        ]
    }
)

print(server_result["messages"][-1].content)

## Quick recap

| Runtime feature | Use it for | Access |
|---|---|---|
| Stream writer | Live progress from a tool | `runtime.stream_writer` |
| Execution info | Thread/run/attempt metadata | `runtime.execution_info` |
| Server info | LangGraph Server metadata | `runtime.server_info` |

### Remember
- Stream writer sends custom progress events; use `stream_mode="custom"` to consume them.
- Execution info describes the current execution.
- Server info is environment-dependent and may be `None` locally.

Official references:
- [LangChain Tools — Runtime context](https://docs.langchain.com/oss/python/langchain/tools#access-context)
- [LangGraph Streaming](https://docs.langchain.com/oss/python/langgraph/streaming)
- [LangGraph Server](https://docs.langchain.com/langsmith/architecture)